In [1]:
import pandas as pd
import duckdb
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_parquet(Path("../data") / "online_retail_II_raw.parquet")
print(df.shape)

(1067371, 9)


In [2]:
%pwd

'C:\\Users\\kanak\\retail-customer-analytics\\notebooks'

In [3]:
%pwd

'C:\\Users\\kanak\\retail-customer-analytics\\notebooks'

In [4]:
from pathlib import Path

data_dir = Path("..") / "data"
print(Path.cwd())
print(data_dir.resolve())
print(list(data_dir.glob("*")))

C:\Users\kanak\retail-customer-analytics\notebooks
C:\Users\kanak\retail-customer-analytics\data
[WindowsPath('../data/clean_sales.parquet'), WindowsPath('../data/cohort_retention.csv'), WindowsPath('../data/excluded_rows.parquet'), WindowsPath('../data/monthly_summary.csv'), WindowsPath('../data/online_retail_II.xlsx'), WindowsPath('../data/online_retail_II_raw.parquet'), WindowsPath('../data/rfm_segments.csv')]


In [5]:
from pathlib import Path

print(Path.cwd())
print(list((Path("..") / "data").glob("*")))

C:\Users\kanak\retail-customer-analytics\notebooks
[WindowsPath('../data/clean_sales.parquet'), WindowsPath('../data/cohort_retention.csv'), WindowsPath('../data/excluded_rows.parquet'), WindowsPath('../data/monthly_summary.csv'), WindowsPath('../data/online_retail_II.xlsx'), WindowsPath('../data/online_retail_II_raw.parquet'), WindowsPath('../data/rfm_segments.csv')]


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  string        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[ns]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  object        
 8   source_sheet  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4), string(1)
memory usage: 73.3+ MB


In [7]:
import pandas as pd
import duckdb
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_parquet(Path("../data") / "online_retail_II_raw.parquet")
print(df.shape)

(1067371, 9)


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  string        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[ns]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  object        
 8   source_sheet  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4), string(1)
memory usage: 73.3+ MB


In [9]:
(df.isna().mean() * 100).round(2)

Invoice         0.00
StockCode       0.00
Description     0.41
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    22.77
Country         0.00
source_sheet    0.00
dtype: float64

In [10]:
df[["Quantity", "Price"]].describe()

,Quantity,Price
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


In [11]:
df.groupby("source_sheet")["InvoiceDate"].agg(["min", "max", "count"])

,min,max,count
source_sheet,,,
Year 2009-2010,2009-12-01 07:45:00,2010-12-09 20:01:00,525461
Year 2010-2011,2010-12-01 08:26:00,2011-12-09 12:50:00,541910


In [12]:
df.drop(columns="source_sheet").duplicated().sum()

np.int64(34335)

In [13]:
is_cancel = df["Invoice"].str.startswith("C")

print("Cancellation invoices (start with C):", is_cancel.sum())
print("Quantity < 0:", (df["Quantity"] < 0).sum())
print("Price <= 0:", (df["Price"] <= 0).sum())
print("Quantity < 0 but NOT a C invoice:", ((df["Quantity"] < 0) & ~is_cancel).sum())

Cancellation invoices (start with C): 19494
Quantity < 0: 22950
Price <= 0: 6207
Quantity < 0 but NOT a C invoice: 3457


In [14]:
duckdb.sql("SELECT COUNT(*) AS n_rows FROM df").show()

┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 1067371 │
└─────────┘



In [15]:
overlap_start = pd.Timestamp("2010-12-01")
overlap_end = pd.Timestamp("2010-12-10")

in_overlap = (df["InvoiceDate"] >= overlap_start) & (df["InvoiceDate"] < overlap_end)
print(df[in_overlap].groupby("source_sheet").size())

s1 = set(df.loc[in_overlap & (df["source_sheet"] == "Year 2009-2010"), "Invoice"])
s2 = set(df.loc[in_overlap & (df["source_sheet"] == "Year 2010-2011"), "Invoice"])
print("Invoices in sheet 1 window:", len(s1))
print("Invoices in sheet 2 window:", len(s2))
print("Invoices in both:", len(s1 & s2))

source_sheet
Year 2009-2010    22523
Year 2010-2011    22523
dtype: int64
Invoices in sheet 1 window: 1088
Invoices in sheet 2 window: 1088
Invoices in both: 1088


In [16]:
cols = [c for c in df.columns if c != "source_sheet"]

dup_any = df.duplicated(subset=cols, keep="first")
dup_same_sheet = df.duplicated(subset=cols + ["source_sheet"], keep="first")

print("Duplicates overall:", dup_any.sum())
print("Duplicates within one sheet:", dup_same_sheet.sum())
print("Duplicates only because the other sheet repeats the row:", (dup_any & ~dup_same_sheet).sum())

Duplicates overall: 34335
Duplicates within one sheet: 12133
Duplicates only because the other sheet repeats the row: 22202


In [17]:
neg_price = df[df["Price"] < 0]
print("Negative price rows:", len(neg_price))
display(neg_price[["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate"]])

zero_price = df[df["Price"] == 0]
print("Zero price rows:", len(zero_price))
print("  with missing Description:", zero_price["Description"].isna().sum())
print("  with negative Quantity:", (zero_price["Quantity"] < 0).sum())
print("  with no Customer ID:", zero_price["Customer ID"].isna().sum())

Negative price rows: 5


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate
179403,A506401,B,Adjust bad debt,1,"-53,594.36",2010-04-29 13:36:00
276274,A516228,B,Adjust bad debt,1,"-44,031.79",2010-07-19 11:24:00
403472,A528059,B,Adjust bad debt,1,"-38,925.87",2010-10-20 12:04:00
825444,A563186,B,Adjust bad debt,1,"-11,062.06",2011-08-12 14:51:00
825445,A563187,B,Adjust bad debt,1,"-11,062.06",2011-08-12 14:52:00


Zero price rows: 6202
  with missing Description: 4382
  with negative Quantity: 3457
  with no Customer ID: 6131


In [18]:
    show = ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "Customer ID"]

print("Quantity of 10,000 or more (either sign):")
display(df.loc[df["Quantity"].abs() >= 10000, show])

print("Five highest prices:")
display(df.nlargest(5, "Price")[show])

Quantity of 10,000 or more (either sign):


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,2010-02-15 11:57:00,"13,902.00"
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,2010-03-17 13:09:00,"13,902.00"
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,2010-03-17 13:09:00,"13,902.00"
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,2010-03-17 13:09:00,"13,902.00"
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,2010-03-17 13:09:00,"13,902.00"
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2010-03-23 15:36:00,"17,940.00"
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2010-03-23 15:36:00,"17,940.00"
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2010-03-23 15:36:00,"17,940.00"
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,2010-03-23 15:36:00,"17,940.00"
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,0.00,2010-05-10 14:55:00,NaN


Five highest prices:


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
748142,C556445,M,Manual,-1,"38,970.00",2011-06-10 15:31:00,"15,098.00"
241824,C512770,M,Manual,-1,"25,111.09",2010-06-17 16:52:00,"17,399.00"
241827,512771,M,Manual,1,"25,111.09",2010-06-17 16:53:00,NaN
320581,C520667,BANK CHARGES,Bank Charges,-1,"18,910.69",2010-08-27 13:42:00,NaN
1050063,C580605,AMAZONFEE,AMAZON FEE,-1,"17,836.46",2011-12-05 11:36:00,NaN


In [19]:
odd_neg = df[(df["Quantity"] < 0) & ~is_cancel]
print("Negative quantity, not a C invoice:", len(odd_neg))
print("  with Price = 0:", (odd_neg["Price"] == 0).sum())
print("  with no Customer ID:", odd_neg["Customer ID"].isna().sum())
display(odd_neg[show].head(10))

print("C invoices with Quantity >= 0:")
display(df.loc[is_cancel & (df["Quantity"] >= 0), show])

Negative quantity, not a C invoice: 3457
  with Price = 0: 3457
  with no Customer ID: 3457


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
263,489464,21733,85123a mixed,-96,0.00,2009-12-01 10:52:00,NaN
283,489463,71477,short,-240,0.00,2009-12-01 10:52:00,NaN
284,489467,85123A,21733 mixed,-192,0.00,2009-12-01 10:53:00,NaN
470,489521,21646,<NA>,-50,0.00,2009-12-01 11:44:00,NaN
3114,489655,20683,<NA>,-44,0.00,2009-12-01 17:26:00,NaN
3162,489660,35956,lost,-1043,0.00,2009-12-01 17:43:00,NaN
3168,489663,35605A,damages,-117,0.00,2009-12-01 18:02:00,NaN
4296,489806,18010,<NA>,-770,0.00,2009-12-02 12:42:00,NaN
4538,489820,21133,invcd as 84879?,-720,0.00,2009-12-02 13:23:00,NaN
4566,489821,85049G,<NA>,-240,0.00,2009-12-02 13:25:00,NaN


C invoices with Quantity >= 0:


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
76799,C496350,M,Manual,1,373.57,2010-02-01 08:24:00,NaN


In [20]:
cols = [c for c in df.columns if c != "source_sheet"]

a = df[in_overlap & (df["source_sheet"] == "Year 2009-2010")][cols].sort_values(cols).reset_index(drop=True)
b = df[in_overlap & (df["source_sheet"] == "Year 2010-2011")][cols].sort_values(cols).reset_index(drop=True)

print("Same shape:", a.shape == b.shape)
print("Identical after sorting:", a.equals(b))

Same shape: True
Identical after sorting: True


In [21]:
keep_mask = ~((df["source_sheet"] == "Year 2010-2011") & (df["InvoiceDate"] < "2010-12-10"))
df_nodup = df[keep_mask].copy()

print("Rows after removing overlap:", len(df_nodup))
print("Exact duplicates still remaining:", df_nodup.drop(columns="source_sheet").duplicated().sum())

Rows after removing overlap: 1044848
Exact duplicates still remaining: 11812


In [22]:
df_nodup["LineValue"] = df_nodup["Quantity"] * df_nodup["Price"]

non_product = df_nodup[~df_nodup["StockCode"].str.match(r"^\d{5}")]

(non_product.groupby("StockCode")
    .agg(rows=("Invoice", "size"), example=("Description", "first"), net_value=("LineValue", "sum"))
    .sort_values("rows", ascending=False)
    .head(25))

,rows,example,net_value
StockCode,,,
POST,2086,POSTAGE,"110,430.41"
DOT,1425,DOTCOM POSTAGE,"309,844.10"
M,1398,Manual,"-82,950.62"
C2,277,CARRIAGE,"13,136.00"
D,173,Discount,"-12,879.63"
S,102,SAMPLES,"-6,000.85"
BANK CHARGES,100,Bank Charges,"-35,482.25"
ADJUST,67,Adjustment by john on 26/01/2010 16,"6,835.24"
AMAZONFEE,36,AMAZON FEE,"-221,520.50"


In [23]:
sales = df_nodup[(df_nodup["Quantity"] > 0) & (df_nodup["Price"] > 0)
                 & ~df_nodup["Invoice"].str.startswith("C")]

by_known = sales.groupby(sales["Customer ID"].notna())["LineValue"].sum()
print(by_known)
print((by_known / by_known.sum() * 100).round(2))

Customer ID
False    3,102,190.74
True    17,431,551.18
Name: LineValue, dtype: float64
Customer ID
False   15.11
True    84.89
Name: LineValue, dtype: float64


In [24]:
cols = [c for c in df_nodup.columns if c not in ("source_sheet", "LineValue")]

dup_extra = df_nodup.duplicated(subset=cols, keep="first")
extra = df_nodup[dup_extra]

print("Extra duplicate rows:", len(extra))
print("Distinct invoices affected:", extra["Invoice"].nunique())
print("Value of the extra copies (qty x price):", round(extra["LineValue"].sum(), 2))
print("Extra copies with no Customer ID:", extra["Customer ID"].isna().sum())
print("Extra copies with zero price:", (extra["Price"] == 0).sum())
print("Extra copies on C invoices:", extra["Invoice"].str.startswith("C").sum())

lines_per_invoice = df_nodup.groupby("Invoice").size()
dups_per_invoice = extra.groupby("Invoice").size()
share = (dups_per_invoice / lines_per_invoice.loc[dups_per_invoice.index]).sort_values(ascending=False)
print("Invoices where at least half the lines are repeats:", (share >= 0.5).sum())

display(df_nodup[df_nodup["Invoice"].isin(share.head(3).index)][show].head(20))

Extra duplicate rows: 11812
Distinct invoices affected: 4387
Value of the extra copies (qty x price): 54228.42
Extra copies with no Customer ID: 136
Extra copies with zero price: 10
Extra copies on C invoices: 61
Invoices where at least half the lines are repeats: 12


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
501989,C536210,22838,3 TIER CAKE TIN RED AND CREAM,-1,14.95,2010-11-30 13:27:00,"17,402.00"
501990,C536210,22838,3 TIER CAKE TIN RED AND CREAM,-1,14.95,2010-11-30 13:27:00,"17,402.00"
682229,C550165,22962,JAM JAR WITH PINK LID,-1,0.85,2011-04-14 16:34:00,"15,358.00"
682230,C550165,22962,JAM JAR WITH PINK LID,-1,0.85,2011-04-14 16:34:00,"15,358.00"
939599,572344,M,Manual,456,1.50,2011-10-24 10:43:00,"14,607.00"
939600,572344,M,Manual,48,1.50,2011-10-24 10:43:00,"14,607.00"
939601,572344,M,Manual,48,1.50,2011-10-24 10:43:00,"14,607.00"
939602,572344,M,Manual,48,1.50,2011-10-24 10:43:00,"14,607.00"
939603,572344,M,Manual,48,1.50,2011-10-24 10:43:00,"14,607.00"
939604,572344,M,Manual,48,1.50,2011-10-24 10:43:00,"14,607.00"


In [25]:
codes = (non_product.groupby("StockCode")
    .agg(rows=("Invoice", "size"), example=("Description", "first"), net_value=("LineValue", "sum"))
    .sort_values("rows", ascending=False))
print("Distinct non-standard codes:", len(codes))
display(codes.iloc[25:])

prefix = df_nodup["Invoice"].str.extract(r"^([A-Za-z]*)")[0]
print(prefix.value_counts())

display(df_nodup[df_nodup["StockCode"] == "B"][show])

Distinct non-standard codes: 62


,rows,example,net_value
StockCode,,,
DCGS0004,5,HAYNES CAMPER SHOULDER BAG,67.94
DCGS0066N,4,NAVY CUDDLES DOG HOODIE,17.30
gift_0001_80,4,Dotcomgiftshop Gift Voucher £80.00,0.00
DCGS0072,4,CAT CAMOUFLAGUE COLLAR,20.68
ADJUST2,3,Adjustment by Peter on Jun 25 2010,731.05
gift_0001_70,3,Dotcomgiftshop Gift Voucher £70.00,59.57
DCGS0068,3,DOGS NIGHT COLLAR,17.12
DCGS0070,3,CAMOUFLAGE DOG COLLAR,25.44
SP1002,3,KID'S CHALKBOARD/EASEL,14.75


0
     1025677
C      19165
A          6
Name: count, dtype: int64


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID
179403,A506401,B,Adjust bad debt,1,"-53,594.36",2010-04-29 13:36:00,NaN
276274,A516228,B,Adjust bad debt,1,"-44,031.79",2010-07-19 11:24:00,NaN
403472,A528059,B,Adjust bad debt,1,"-38,925.87",2010-10-20 12:04:00,NaN
825443,A563185,B,Adjust bad debt,1,"11,062.06",2011-08-12 14:50:00,NaN
825444,A563186,B,Adjust bad debt,1,"-11,062.06",2011-08-12 14:51:00,NaN
825445,A563187,B,Adjust bad debt,1,"-11,062.06",2011-08-12 14:52:00,NaN
